# GSE122149 — T-cell State Scoring

Exploratory signature-scoring workflow for CD8 T-cell transcriptional states.

> **Note:** Scores are relative within this dataset. They are not absolute cell phenotype labels.

## 0. Imports & paths

In [1]:
import sys
from pathlib import Path

ROOT = Path("..")
sys.path.insert(0, str(ROOT / "src"))

from load_data import load_counts, load_series_matrix
from metadata import build_metadata, validate_samples
from scoring import load_signatures, log2_transform, zscore_genes, score_signatures, summarize_scores_by_group
from plotting import set_plot_style, plot_pca, plot_signature_scores, plot_signature_heatmap, plot_grouped_heatmap, plot_marker_heatmap, plot_clustermap

set_plot_style()

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGS      = ROOT / "outputs" / "figures"
TABLES    = ROOT / "outputs" / "tables"
CONFIG    = ROOT / "config" / "signatures.yml"

## 1. Load expression matrix

In [2]:
counts = load_counts(RAW / "GSE122149_Smanes_normalized_counts.txt.gz")
print(counts.shape)  # (genes, samples)
counts.head()

(15038, 21)


,39E76hSINACT,46E86hSINACT,53E96hSINACT,33E76hACT,40E86hACT,47E96hACT,34E724hACT,41E824hACT,48E924hACT,35E748hACT,...,49E948hACT,36E76hPDL1,43E86hPDL1,50E96hPDL1,37E724hPDL1,44E824hPDL1,51E924hPDL1,38E748hPDL1,45E848hPDL1,52E948hPDL1
gene,,,,,,,,,,,,,,,,,,,,,
7SK,3.562591,4.284355,5.237833,2.890315,3.707555,3.623706,0.816940,0.871556,2.539618,1.722823,...,1.844777,1.776241,4.284376,6.027480,2.390034,2.440874,4.070535,1.617917,4.085358,3.261046
A1BG,7.125182,5.141226,3.491888,4.817191,4.634443,3.623706,4.901638,6.100890,5.925775,6.029879,...,2.767165,5.328723,4.284376,4.305343,5.576746,4.881748,3.256428,5.662709,4.902430,4.891569
A1BG-AS1,13.359716,13.709936,12.221609,11.561259,1.853777,1.811853,12.254094,13.073336,1.693078,12.921169,...,8.301496,15.098048,12.853129,12.054960,2.390034,13.831620,13.839820,17.797085,13.073145,15.489969
A2M-AS1,4.453239,5.141226,6.983777,3.853753,4.634443,6.341485,3.267758,1.743111,3.386157,1.722823,...,1.844777,3.552482,5.141252,6.888548,7.170102,1.627249,11.397499,8.898542,10.621931,12.228923
AAAS,13.359716,18.851162,1.745944,14.451574,15.757107,15.400749,23.691248,28.761340,28.782334,4.307056,...,40.585094,15.098048,22.278757,16.360302,17.526915,21.967867,22.794997,19.415001,22.878005,20.381538


## 2. Load & build sample metadata

In [8]:
series = load_series_matrix(RAW / "GSE122149_series_matrix.txt.gz")
meta = build_metadata(series)
meta.head()

,geo_accession,title,timepoint,condition
sample_id,,,,
33E76hACT,GSM3456159,"33E76hACT: E7 Activated CD8+ T cells, time poi...",6h,activated_CD3_CD28
34E724hACT,GSM3456160,"34E724hACT: E7 Activated CD8+ T cells, time po...",24h,activated_CD3_CD28
35E748hACT,GSM3456161,"35E748hACT: E7 Activated CD8+ T cells, time po...",48h,activated_CD3_CD28
36E76hPDL1,GSM3456162,"36E76hPDL1: E7 Activated+PDL1 CD8+ T cells, ti...",6h,activated_PD1_PDL1
37E724hPDL1,GSM3456163,"37E724hPDL1: E7 Activated+PDL1 CD8+ T cells, t...",24h,activated_PD1_PDL1


## 3. Validate sample IDs

In [ ]:
validate_samples(counts, meta)

## 4. log2(x+1) transform

In [9]:
expr = log2_transform(counts)

## 5. PCA

PCA is an unsupervised sanity check to visualize whether samples separate by stimulation condition, timepoint, or both. It is not used as a classifier.

In [ ]:
# Main figure: color=condition, shape=timepoint (both variables in one plot)
plot_pca(expr, meta, out_path=FIGS / "pca_by_condition_timepoint.png")

# Faceted views
plot_pca(expr, meta, out_path=FIGS / "pca_by_condition.png", color_col="condition", style_col="condition")
plot_pca(expr, meta, out_path=FIGS / "pca_by_timepoint.png",  color_col="timepoint",  style_col="timepoint")

## 6. Load signatures

In [11]:
signatures = load_signatures(CONFIG)
print({k: len(v) for k, v in signatures.items()})

{'memory_tscm': 9, 'effector': 9, 'exhaustion_inhibitory': 7, 'memory_metabolism': 4, 'glycolysis_activation': 5}


## 7. Z-score genes & score signatures

In [12]:
expr_z = zscore_genes(expr)
scores, missing = score_signatures(expr_z, signatures)
print(f"Scored {scores.shape[1]} signatures across {scores.shape[0]} samples")
scores.head()

Scored 5 signatures across 21 samples


,memory_tscm,effector,exhaustion_inhibitory,memory_metabolism,glycolysis_activation
39E76hSINACT,-0.002176,-0.243806,-0.642887,-0.247904,-0.124537
46E86hSINACT,0.329418,-0.596260,-0.119403,0.358633,-0.399855
53E96hSINACT,0.528168,0.163134,-0.380778,0.411977,-0.007154
33E76hACT,0.107657,0.420207,0.381109,0.168371,0.175525
40E86hACT,-0.054888,0.222662,0.712964,0.134062,0.314679


## 8. Save outputs

In [13]:
scores.to_csv(TABLES / "signature_scores.csv")
missing.to_csv(TABLES / "missing_signature_genes.csv", index=False)
print("Missing genes per signature:")
print(missing.groupby("signature")["gene"].apply(list))

Missing genes per signature:
signature
memory_metabolism    [PPARGC1A]
Name: gene, dtype: object


## 9. Signature scores by condition (strip plot)

More interpretable than a heatmap: shows replicate-level variation by condition.

In [ ]:
plot_signature_scores(scores, meta, out_path=FIGS / "signature_scores_by_condition.png")

## 11. Signature score heatmap (per sample)

In [ ]:
plot_signature_heatmap(
    scores, meta,
    out_path=FIGS / "signature_scores_heatmap.png",
    sort_cols=["condition", "timepoint"],
)

## 12. Grouped signature heatmap (condition × timepoint means)

Averaging within groups reduces sample-level noise and often reveals cleaner patterns.

In [ ]:
grouped = summarize_scores_by_group(scores, meta, group_cols=["condition", "timepoint"])
plot_grouped_heatmap(grouped, out_path=FIGS / "signature_scores_grouped_heatmap.png")

## 13. Hierarchical clustering — top 200 variable genes

Dendrogram of samples and genes; columns color-annotated by condition | timepoint.

In [ ]:
plot_clustermap(expr, meta, out_path=FIGS / "clustermap_top200_var_genes.png")

## 14. Marker gene heatmap

Individual gene z-scores reveal what is driving each signature.
The Tscm/memory signature may not dominate in this short-stimulation dataset — that is expected and noted.

In [ ]:
marker_genes = [
    # memory/resting-like
    "TCF7", "KLF2", "CCR7", "IL7R", "LEF1", "SELL",
    # early activation
    "CD69", "IL2RA", "FOS", "JUN", "MYC",
    # effector/cytokine
    "IRF4", "TBX21", "GZMB", "PRF1", "IFNG",
    # inhibitory/checkpoint
    "PDCD1", "LAG3", "CTLA4", "HAVCR2", "TIGIT", "TOX",
    # metabolism
    "SLC2A1", "HIF1A", "LDHA", "CPT1A",
]
plot_marker_heatmap(
    expr_z, meta, genes=marker_genes,
    out_path=FIGS / "marker_gene_heatmap.png",
    sort_cols=["condition", "timepoint"],
)